# Stage 1: Evidence Retrieval for Question Answering
**Authors:** Roberto Herrera and Bach Nguyen

This notebook implements the retrieval stage of a two-stage question-answering pipeline. Given a question and its associated context, Stage 1 divides the source text into chunks, embeds them, and uses a Chroma vector store to retrieve relevant passages. Stage 2 uses an extractive QA model to predict an answer from the selected text.

We compare fixed token windows and sentence-aware chunks at sizes of 96 and 192 tokens using MiniLM-L6, MiniLM-L12, and MPNet. All configurations initially use no overlap and cosine similarity.

Evaluation measures supporting evidence coverage and retrieval latency. Gold annotations are used only for evaluation, and the QA model’s tokenizer determines which retrieved chunks fit in its input.

## 1. Setup

In [6]:
import os
import re
import ast
import json
import time
import random
import platform
import pandas as pd
from datetime import datetime
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

HOTPOTQA_PATH = "/root/.cache/kagglehub/datasets/jeromeblanchet/hotpotqa-question-answering-dataset/versions/1/hotpot_train_v1.1.json"
QASPER_DIR = "/root/.cache/kagglehub/datasets/thedevastator/qasper-nlp-questions-and-evidence/versions/2"

SEED = 42
random.seed(SEED)

MACHINE_LABEL = "laptopRH"   # change this when the notebook runs on another machine

def machine_info():
    with open("/proc/meminfo") as f:
        mem_gb = round(int(f.readline().split()[1]) / 1024 / 1024, 1)
    with open("/proc/cpuinfo") as f:
        cpu = next((line.split(":")[1].strip() for line in f if line.startswith("model name")), "unknown")
    return {
        "label": MACHINE_LABEL,
        "cpu_model": cpu,
        "cpu_cores": os.cpu_count(),
        "memory_gb": mem_gb,
        "platform": platform.platform(),
        "python": platform.python_version(),
    }


MACHINE = machine_info()
print(MACHINE)

RESULTS_DIR = Path("results") / MACHINE_LABEL
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def save_result(name, settings, metrics):
    record = {
        "experiment": name,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "seed": SEED,
        "machine": MACHINE,
        "settings": settings,
        "metrics": metrics,
    }
    (RESULTS_DIR / (name + ".json")).write_text(json.dumps(record, indent=2))
    return record

{'label': 'laptopRH', 'cpu_model': '12th Gen Intel(R) Core(TM) i5-1235U', 'cpu_cores': 12, 'memory_gb': 7.6, 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.35', 'python': '3.10.12'}


## 2. Inspect the datasets

Load HotPotQA and inspect one example to identify its question, context, and supporting facts. List the QASPER files before loading them so we can preserve their original structure and evidence annotations.

In [7]:
from collections import Counter

def parse_cell(s):
    s = re.sub(r",\s*dtype=\w+\)", ")", s)
    s = s.replace("array(", "(")
    return ast.literal_eval(s)

def paper_paragraphs(row):
    ft = parse_cell(row["full_text"])
    return [
        (section, p)
        for section, plist in zip(ft["section_name"], ft["paragraphs"])
        for p in plist
        if p.strip()
    ]

with open(HOTPOTQA_PATH) as f:
    hotpot = json.load(f)

qasper = pd.read_csv(Path(QASPER_DIR) / "train.csv", dtype={"id": str})
paper_lengths = [sum(len(p) for _, p in paper_paragraphs(row)) for _, row in qasper.iterrows()]

print("HotpotQA examples", len(hotpot))
print("QASPER papers", len(qasper))
print(pd.Series(paper_lengths).describe())

HotpotQA examples 90447
QASPER papers 888
count       888.000000
mean      24020.396396
std       14525.773967
min           0.000000
25%       15485.250000
50%       23031.500000
75%       28203.750000
max      167959.000000
dtype: float64


## 3. Retrieval functions

We compare recursive and semantic chunking with single-pass and two-hop retrieval. The requested chunk size is capped at each embedding model’s input limit, including special tokens. Semantic segmentation uses a percentile threshold on adjacent text-unit distances; oversized sentences are split before embedding.

Chunks are indexed in ChromaDB using cosine similarity. HotPotQA uses one index per question’s candidate context, while QASPER reuses one index across a paper’s questions. Two-hop retrieval expands the question with the first retrieved chunk and returns up to k unique chunks in total. Source identifiers are retained for source-coverage evaluation.

In [20]:
from uuid import uuid4

import numpy as np
import spacy
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")


def make_splitter(chunk_size, chunk_overlap, tokenizer, token_limit):
    return RecursiveCharacterTextSplitter(
        chunk_size=min(chunk_size, token_limit),
        chunk_overlap=chunk_overlap,
        length_function=lambda text: len(
            tokenizer.encode(text, add_special_tokens=True, truncation=False)
        ),
        add_start_index=True,
    )


def semantic_chunks(text, metadata, embedder, splitter, percentile):
    """Embed bounded sentence units and split at semantic boundaries."""
    units = []
    for sentence in nlp(text).sents:
        if not sentence.text.strip():
            continue
        for piece in splitter.create_documents([sentence.text]):
            start = sentence.start_char + piece.metadata["start_index"]
            units.append((piece.page_content, start))

    if not units:
        return []

    breaks = []
    if len(units) > 1:
        vectors = np.asarray(
            embedder.embed_documents([text for text, _ in units]),
            dtype=float,
        )
        vectors /= np.maximum(
            np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12
        )
        distances = 1 - np.clip(
            np.sum(vectors[:-1] * vectors[1:], axis=1), -1, 1
        )
        threshold = np.percentile(distances, percentile)
        breaks = [
            i + 1 for i, distance in enumerate(distances)
            if distance > threshold
        ]

    docs = []
    first = 0
    for last in breaks + [len(units)]:
        start = units[first][1]
        end = units[last - 1][1] + len(units[last - 1][0])
        pieces = splitter.create_documents(
            [text[start:end]], metadatas=[metadata]
        )
        for piece in pieces:
            piece.metadata["start_index"] += start
            docs.append(piece)
        first = last

    return docs


def build_store(paragraphs, embedder, chunking, chunk_size,
                chunk_overlap=0, percentile=None):
    """Build one context index using the active embedding model."""
    tokenizer = embedder._client.tokenizer
    token_limit = embedder._client.max_seq_length
    splitter = make_splitter(
        chunk_size, chunk_overlap, tokenizer, token_limit
    )

    if chunking not in {"recursive", "semantic"}:
        raise ValueError("chunking must be 'recursive' or 'semantic'")
    if chunking == "semantic":
        if percentile is None or not 0 < percentile < 100:
            raise ValueError("Semantic percentile must be between 0 and 100.")
        if chunk_overlap != 0:
            raise ValueError("Semantic chunking uses zero overlap.")

    docs = []
    for text, metadata in paragraphs:
        if chunking == "recursive":
            docs.extend(
                splitter.create_documents([text], metadatas=[metadata])
            )
        else:
            docs.extend(
                semantic_chunks(
                    text, metadata, embedder, splitter, percentile
                )
            )

    if not docs:
        return None

    return Chroma.from_documents(
        docs,
        embedder,
        collection_name="retrieval_" + uuid4().hex,
        collection_metadata={"hnsw:space": "cosine"},
    )


def retrieve_two_hop(store, question, k, query_tokenizer, max_query_tokens):
    """Return up to k unique chunks using one two-hop path."""
    if k < 1:
        raise ValueError("k must be at least 1")
    if store is None:
        return [], []

    question_length = len(
        query_tokenizer.encode(question, add_special_tokens=True)
    )
    if question_length > max_query_tokens:
        raise ValueError("Question exceeds the embedding model's token limit.")

    results = store.similarity_search(question, k=1)
    if not results:
        return [], []

    if k > 1:
        evidence = results[0].page_content
        offsets = query_tokenizer(
            evidence,
            add_special_tokens=False,
            return_offsets_mapping=True,
            truncation=False,
        )["offset_mapping"]

        end = min(len(offsets), max_query_tokens - question_length)
        while end > 0:
            query = question + "\n" + evidence[:offsets[end - 1][1]]
            if len(query_tokenizer.encode(
                query, add_special_tokens=True
            )) <= max_query_tokens:
                break
            end -= 1

        if end == 0:
            raise ValueError("No first-hop evidence fits beside the question.")

        for doc in store.similarity_search(query, k=k):
            duplicate = any(
                doc.metadata == previous.metadata
                and doc.page_content == previous.page_content
                for previous in results
            )
            if not duplicate:
                results.append(doc)
            if len(results) == k:
                break

    return (
        [doc.page_content for doc in results],
        [doc.metadata for doc in results],
    )


def retrieve(store, question, k, method,
             query_tokenizer=None, max_query_tokens=None):
    if k < 1:
        raise ValueError("k must be at least 1")
    if method not in {"single_pass", "two_hop"}:
        raise ValueError("method must be 'single_pass' or 'two_hop'")
    if store is None:
        return [], []
    if query_tokenizer is None or max_query_tokens is None:
        raise ValueError("Provide the embedding tokenizer and token limit.")

    if len(query_tokenizer.encode(
        question, add_special_tokens=True
    )) > max_query_tokens:
        raise ValueError("Question exceeds the embedding model's token limit.")

    if method == "two_hop":
        return retrieve_two_hop(
            store, question, k, query_tokenizer, max_query_tokens
        )

    results = store.similarity_search(question, k=k)
    return (
        [doc.page_content for doc in results],
        [doc.metadata for doc in results],
    )


def hotpot_paragraphs(example):
    return [
        (" ".join(sentences), {"title": title})
        for title, sentences in example["context"]
    ]


def qasper_paragraphs(row):
    return [
        (paragraph, {"section": section, "para_id": pid})
        for pid, (section, paragraph) in enumerate(paper_paragraphs(row))
    ]

## 4. Experiment configuration

We compare recursive and semantic chunking with single-pass and two-hop retrieval across three embedding models. Both chunking methods use a shared 192-token size limit and no overlap. Semantic boundaries use the 90th percentile of adjacent sentence distances. Each method retrieves up to four chunks, giving 12 retrieval configurations evaluated separately on both datasets.

In [22]:
configs = [
    {
        "embedding": "sentence-transformers/all-MiniLM-L6-v2",
        "chunking": "recursive",
        "chunk_size": chunk_size,
        "chunk_overlap": 0,
        "method": "single_pass",
        "k": k,
    }
    for chunk_size in [128, 256]
    for k in [1, 3, 5, 10]
]

pd.DataFrame(configs)

,embedding,chunking,chunk_size,chunk_overlap,method,k
0,sentence-transformers/all-MiniLM-L6-v2,recursive,128,0,single_pass,1
1,sentence-transformers/all-MiniLM-L6-v2,recursive,128,0,single_pass,3
2,sentence-transformers/all-MiniLM-L6-v2,recursive,128,0,single_pass,5
3,sentence-transformers/all-MiniLM-L6-v2,recursive,128,0,single_pass,10
4,sentence-transformers/all-MiniLM-L6-v2,recursive,256,0,single_pass,1
5,sentence-transformers/all-MiniLM-L6-v2,recursive,256,0,single_pass,3
6,sentence-transformers/all-MiniLM-L6-v2,recursive,256,0,single_pass,5
7,sentence-transformers/all-MiniLM-L6-v2,recursive,256,0,single_pass,10


## 5. Retrieval evaluation

We measure supporting-source recall and complete-source retrieval. HotPotQA requires coverage of all supporting paragraph titles. QASPER is scored against each annotator’s evidence set, taking the highest recall across valid alternatives. A source is retrieved when at least one returned chunk belongs to it; this measures source coverage rather than complete evidence retention. Questions without answerable text evidence or with unmapped evidence are flagged separately.

In [14]:
def normalize_text(text):
    return " ".join(text.split())


def qasper_questions(row):
    paragraphs = paper_paragraphs(row)
    paragraph_ids = {
        normalize_text(text): pid
        for pid, (_, text) in enumerate(paragraphs)
    }
    qas = parse_cell(row["qas"])

    for question_id, question, answers in zip(
        qas["question_id"], qas["question"], qas["answers"]
    ):
        gold_sets = []
        unmapped = False

        for answer in answers["answer"]:
            if answer["unanswerable"]:
                continue

            evidence = {
                normalize_text(text)
                for text in answer["evidence"]
                if text.strip()
            }
            if not evidence:
                continue

            if not evidence.issubset(paragraph_ids):
                unmapped = True
            else:
                gold_sets.append({paragraph_ids[text] for text in evidence})

        status = (
            "unmapped_evidence" if unmapped
            else "ok" if gold_sets
            else "no_answerable_text_evidence"
        )
        yield {
            "question_id": question_id,
            "question": question,
            "gold_sets": gold_sets,
            "evidence_status": status,
        }


def source_metrics(metadata, gold_sets, source_key):
    if not gold_sets:
        return {"source_recall": None, "all_sources": None}

    retrieved = {item[source_key] for item in metadata}
    recall = max(
        len(retrieved & gold) / len(gold)
        for gold in gold_sets
    )
    return {
        "source_recall": recall,
        "all_sources": int(recall == 1.0),
    }

## 6. Generate retrieval candidates

For each chunk size, we build one index per HotPotQA context or QASPER paper and retrieve up to ten ranked chunks per question. The saved rankings support subsequent QA comparisons using the first 1, 3, 5, or 10 candidates. Index construction and top-10 retrieval latency are recorded separately. Reference answers remain in the original datasets and are joined through question and context identifiers.

In [24]:
import traceback

LIMIT = 20
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CONFIDENCE_THRESHOLD = 0.5

QA_DIR = RESULTS_DIR / (
    "qa_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
QA_DIR.mkdir(parents=True, exist_ok=True)

started = synced_time()
embedder = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": DEVICE},
)
embedding_load_s = synced_time() - started
embedder.embed_query("Model warm-up.")


def evaluation_contexts(dataset):
    if dataset == "hotpot":
        for example in hotpot[:LIMIT]:
            yield (
                example["_id"],
                hotpot_paragraphs(example),
                [(example["_id"], example["question"])],
            )
    else:
        for _, row in qasper.iloc[:LIMIT].iterrows():
            qas = parse_cell(row["qas"])
            yield (
                row["id"],
                qasper_paragraphs(row),
                list(zip(qas["question_id"], qas["question"])),
            )


(QA_DIR / "settings.json").write_text(json.dumps({
    "qa_model": QA_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "device": DEVICE,
    "seed": SEED,
    "limit": LIMIT,
    "chunking": "recursive",
    "chunk_overlap": 0,
    "retrieval_method": "single_pass",
    "chunk_sizes": [128, 256],
    "depths": [1, 3, 5, 10],
    "reader_limit": 512,
    "max_answer_len": 512,
    "handle_impossible_answer": True,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "qa_model_load_s": model_load_s,
    "embedding_model_load_s": embedding_load_s,
    "timing_mode": "live_query_with_prebuilt_collection",
}, indent=2))

rows = []
output_path = QA_DIR / "predictions.jsonl"

with output_path.open("w", encoding="utf-8") as output:
    for chunk_size in [128, 256]:
        for dataset in ["hotpot", "qasper"]:
            for context_id, paragraphs, questions in evaluation_contexts(dataset):
                store = None
                build_times = {}
                build_error = None

                try:
                    store = build_store(
                        paragraphs, embedder, "recursive", chunk_size,
                        chunk_overlap=0, timings=build_times,
                    )
                except Exception:
                    build_error = traceback.format_exc()

                try:
                    for question_id, question in questions:
                        gold = references[
                            (dataset, str(context_id), str(question_id))
                        ]

                        for k in [1, 3, 5, 10]:
                            record = {
                                "dataset": dataset,
                                "context_id": str(context_id),
                                "question_id": str(question_id),
                                "question": question,
                                "embedding": EMBEDDING_MODEL,
                                "chunking": "recursive",
                                "chunk_overlap": 0,
                                "method": "single_pass",
                                "chunk_size": chunk_size,
                                "k": k,
                                "qa_model": QA_MODEL,
                                "answer_type": gold["answer_type"],
                                "reference_answers": gold["answers"],
                                "status": "ok",
                                "timing_mode": "live_query_with_prebuilt_collection",
                                "timings": dict(build_times),
                            }

                            try:
                                if build_error is not None:
                                    raise RuntimeError(build_error)

                                record["empty_context"] = store is None
                                started = synced_time()

                                retrieved = retrieve(
                                    store, question, k, "single_pass",
                                    embedder._client.tokenizer,
                                    embedder._client.max_seq_length,
                                    timings=record["timings"],
                                )
                                packing_started = synced_time()
                                packed = pack_context(
                                    question, retrieved["chunks"],
                                    reader.tokenizer,
                                    timings=record["timings"],
                                )
                                prediction = answer_question(
                                    question, packed, record["timings"]
                                )
                                finished = synced_time()

                                record["timings"].update({
                                    "packing_qa_s": finished - packing_started,
                                    "query_to_answer_s": finished - started,
                                })

                                assert packed["input_tokens"] <= 512
                                if prediction["answer"]:
                                    assert packed["context"][
                                        prediction["start"]:prediction["end"]
                                    ] == prediction["answer"]

                                record.update({
                                    "retrieval": retrieved,
                                    "prediction": prediction,
                                    "packed": packed,
                                    "confidence_pass": (
                                        prediction["score"]
                                        >= CONFIDENCE_THRESHOLD
                                    ),
                                    **score_answer(
                                        prediction["answer"], gold["answers"]
                                    ),
                                })

                            except Exception:
                                record.update({
                                    "status": "error",
                                    "error": traceback.format_exc(),
                                    "exact_match": (
                                        0.0 if gold["answers"] else None
                                    ),
                                    "f1": 0.0 if gold["answers"] else None,
                                    "confidence_pass": False,
                                })

                            output.write(
                                json.dumps(record, ensure_ascii=False) + "\n"
                            )
                            output.flush()

                            row = {
                                name: record[name] for name in [
                                    "dataset", "context_id", "question_id",
                                    "chunk_size", "k", "answer_type", "status",
                                    "exact_match", "f1", "confidence_pass",
                                ]
                            }
                            row.update(record["timings"])
                            if record["status"] != "ok":
                                row["packing_qa_s"] = None
                                row["query_to_answer_s"] = None
                            rows.append(row)

                            if len(rows) % 40 == 0:
                                print(
                                    len(rows), "evaluations completed | errors:",
                                    sum(r["status"] == "error" for r in rows),
                                    flush=True,
                                )
                finally:
                    if store is not None:
                        store.delete_collection()

            print("Finished:", dataset, chunk_size, flush=True)

qa_results = pd.DataFrame(rows)
qa_results.to_csv(QA_DIR / "question_metrics.csv", index=False)

qa_summary = qa_results.groupby(["dataset", "chunk_size", "k"]).agg(
    questions=("status", "size"),
    errors=("status", lambda values: (values == "error").sum()),
    scored_questions=("exact_match", "count"),
    exact_match=("exact_match", "mean"),
    f1=("f1", "mean"),
    confidence_passes=("confidence_pass", "sum"),
    packing_qa_mean_s=("packing_qa_s", "mean"),
    query_to_answer_mean_s=("query_to_answer_s", "mean"),
    query_to_answer_p95_s=(
        "query_to_answer_s", lambda values: values.quantile(0.95)
    ),
).reset_index()

qa_summary.to_csv(QA_DIR / "summary.csv", index=False)
display(qa_summary.round(4))
print("Results saved in:", QA_DIR)

NameError: name 'synced_time' is not defined